In [ ]:
# Parameters
# ------ Configuration Parameters (can be overridden by papermill) ------
TICKER = "AAPL"                  # single stock to evaluate
MODEL_NAME = "mamba"             # single model: gru, transformer, gru_transformer, or mamba
ABLATION_SEED = 4718
PREPROCESSING_CONTRACT_VERSION = 1
MAX_SAMPLES_PER_ORDER = 100
SPLIT_CAP_RANDOM_SEED = 4718
NUM_TIME_STEPS = 30
BATCH_SIZE = 512
ARTIFACT_DIR = "/ocean/projects/cis260122p/shared/artifacts"
# -----------------------------------------------------------------------


# Toxicity Feature Ablation — Permutation Importance

Runs **permutation feature importance** on the held-out test set for every trained model artifact
found in `ARTIFACT_DIR/{TICKER}/`.

**Method**: For each feature column `j`, shuffle its values across all test samples, run inference,
and measure the drop in the primary metrics (weighted C-td and weighted PR-AUC for TOXIC_FILL).
A large positive drop means the model relies heavily on that feature.

**Memory strategy**: The full test set (~1.7M samples × 500 steps × 34 features) is never
materialised at once. Instead:
- Baseline CIF `(K, T, N)` is collected in one streaming pass (~400 MB).
- For each permutation, only one feature strip `(N, seq_len)` is extracted (~3.4 GB),
  shuffled, then injected during a second streaming inference pass.

Feature layout (34 total):
- `lob_00`–`lob_19`: LOB snapshot features
- `tox_*`: 12 toxicity features
- `side`: order side (bid=1 / ask=0)
- `mask`: valid-timestep mask


In [2]:
from __future__ import annotations

import gc
import json
import os
import pickle
import random
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Discover project root
for _candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (_candidate / "src").exists():
        repo_root = _candidate.resolve()
        if str(repo_root) not in sys.path:
            sys.path.insert(0, str(repo_root))
        break
else:
    raise RuntimeError("Could not locate project root containing 'src'.")

import torch
from sklearn.metrics import auc, precision_recall_curve
import wandb
from dotenv import find_dotenv, load_dotenv

from src.models import (
    DeepHitMambaCompeting,
    DeepHitRNNCompeting,
    DeepHitRNNTransformerCompeting,
    DeepHitTransformerCompeting,
)
from src.notebook_data import (
    DynamicSampleManifest,
    apply_dynamic_normalizer,
    materialize_dynamic_samples_from_manifest,
    select_manifest_indices_by_source_rows,
)

warnings.filterwarnings("ignore", category=FutureWarning)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", device)

if device.type == "cpu":
    torch.set_num_threads(max(1, os.cpu_count() or 1))


def _set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


_set_seed(ABLATION_SEED)
rng = np.random.default_rng(ABLATION_SEED)


## Feature Names

In [ ]:
LOB_DIM = 20
TOX_DIM = 12

LOB_FEATURE_NAMES = [f"lob_{i:02d}" for i in range(LOB_DIM)]

# From src/features/compose.py ToxicityFeatures.get_feature_names()
TOX_FEATURE_NAMES = [
    "spread_relative",
    "imbalance_tob",
    "depth_imbalance",
    "weighted_imbalance",
    "total_weighted_volume",
    "tob_concentration",
    "volume_cv",
    "microprice_offset",
    "significant_bid_levels",
    "significant_ask_levels",
    "time_delta_ms",
    "queue_position",
]

ALL_FEATURE_NAMES = LOB_FEATURE_NAMES + TOX_FEATURE_NAMES + ["side", "mask"]
FEATURE_DIM = len(ALL_FEATURE_NAMES)  # 34
FEATURE_GROUPS = ["LOB"] * LOB_DIM + ["Toxicity"] * TOX_DIM + ["Meta", "Meta"]

# Only these 12 features are permuted
ABLATION_FEATURE_NAMES = TOX_FEATURE_NAMES

print(f"Total features: {FEATURE_DIM}")
print(f"Features to permute ({len(ABLATION_FEATURE_NAMES)}): {ABLATION_FEATURE_NAMES}")


## Load Dataset

In [4]:
DATASET_BASE_DIR = Path("/ocean/projects/cis260122p/shared/data/datasets")
DATASET_STEM_TEMPLATE = "labeled_dataset_XNAS_ITCH_{ticker}_mbo_20251001_20260101"

REQUIRED_RUNTIME_ARRAYS = [
    "train_sample_idx", "val_sample_idx",
    "Y_train_disc", "D_train_disc",
    "Y_val", "D_val", "Y_val_disc", "D_val_disc",
    "ORDER_KEYS_TRAIN", "ORDER_KEYS_VAL",
    "UPDATE_IDX_TRAIN", "UPDATE_IDX_VAL",
    "feat_mean", "feat_std", "time_grid",
]

ticker_symbol = str(TICKER).upper()
stem = DATASET_STEM_TEMPLATE.format(ticker=ticker_symbol)

paths = {
    "dynamic_order_store":    DATASET_BASE_DIR / f"{stem}_dynamic_order_store.pkl",
    "dynamic_manifest":       DATASET_BASE_DIR / f"{stem}_dynamic_sample_manifest.parquet",
    "dynamic_manifest_meta":  DATASET_BASE_DIR / f"{stem}_dynamic_manifest_meta.json",
    "dynamic_preprocessed":   DATASET_BASE_DIR / f"{stem}_dynamic_preprocessed.npz",
}
missing = [str(p) for p in paths.values() if not p.exists()]
if missing:
    raise FileNotFoundError(f"[{ticker_symbol}] Missing files:\n" + "\n".join(missing))

with open(paths["dynamic_order_store"], "rb") as f:
    ORDER_STORE = pickle.load(f)

manifest_df = pd.read_parquet(paths["dynamic_manifest"])
SAMPLE_MANIFEST = DynamicSampleManifest(
    order_ptr=manifest_df["order_ptr"].to_numpy(dtype=np.int32, copy=False),
    end_idx=manifest_df["end_idx"].to_numpy(dtype=np.int32, copy=False),
    y=manifest_df["y"].to_numpy(dtype=np.float32, copy=False),
    d=manifest_df["d"].to_numpy(dtype=np.int64, copy=False),
    order_ids=manifest_df["order_ids"].to_numpy(dtype=np.int64, copy=False),
    entry_times=manifest_df["entry_times"].to_numpy(dtype=np.int64, copy=False),
    source_row_idx=manifest_df["source_row_idx"].to_numpy(dtype=np.int64, copy=False),
    update_idx=manifest_df["update_idx"].to_numpy(dtype=np.int32, copy=False),
)

with open(paths["dynamic_manifest_meta"], "r", encoding="utf-8") as f:
    manifest_meta = json.load(f)

with np.load(paths["dynamic_preprocessed"], allow_pickle=False) as npz:
    runtime_npz = {k: npz[k] for k in npz.files}

# Contract checks (same as quick notebook)
contract_version = int(manifest_meta.get("preprocessing_contract_version", -1))
if contract_version != PREPROCESSING_CONTRACT_VERSION:
    raise ValueError(f"Contract mismatch: expected={PREPROCESSING_CONTRACT_VERSION}, found={contract_version}")

cfg = manifest_meta.get("preprocessing_config", {})
if int(cfg.get("max_samples_per_source_row", -1)) != MAX_SAMPLES_PER_ORDER:
    raise ValueError("max_samples_per_source_row mismatch")
if int(cfg.get("split_cap_random_seed", -1)) != SPLIT_CAP_RANDOM_SEED:
    raise ValueError("split_cap_random_seed mismatch")
if int(cfg.get("num_time_steps", -1)) != NUM_TIME_STEPS:
    raise ValueError("num_time_steps mismatch")

feat_mean = runtime_npz["feat_mean"].astype(np.float32)
feat_std  = runtime_npz["feat_std"].astype(np.float32)
time_grid = runtime_npz["time_grid"].astype(np.float32)

n_train = int(manifest_meta["n_train_orders"])
n_val   = int(manifest_meta["n_val_orders"])
n_test  = int(manifest_meta["n_test_orders"])

test_source_rows = np.arange(n_train + n_val, n_train + n_val + n_test, dtype=np.int64)
test_sample_idx  = select_manifest_indices_by_source_rows(SAMPLE_MANIFEST, test_source_rows)

if test_sample_idx.size == 0:
    raise ValueError(f"[{ticker_symbol}] No test samples found.")

# Use source_row_idx as the order key — matches the quick notebook's ORDER_KEYS_TEST.
# Using order_ids instead can give different per-sample weights if any order_id spans
# multiple source rows (e.g. order amendments or re-entries with the same exchange ID).
idx = test_sample_idx.astype(np.int64)
Y_test          = SAMPLE_MANIFEST.y[idx].astype(np.float32)
D_test          = SAMPLE_MANIFEST.d[idx].astype(np.int64)
ORDER_KEYS_TEST = SAMPLE_MANIFEST.source_row_idx[idx].astype(np.int64)
UPDATE_IDX_TEST = SAMPLE_MANIFEST.update_idx[idx].astype(np.int64)

FEATURE_DIM_TOTAL = int(ORDER_STORE.lob_dim + ORDER_STORE.tox_dim + 2)
SEQUENCE_LENGTH   = int(ORDER_STORE.lookback_steps)
output_steps      = int(time_grid.shape[0])

order_counts = pd.Series(ORDER_KEYS_TEST).value_counts()
ORDER_TEST_WEIGHTS = np.array(
    [1.0 / order_counts[int(k)] for k in ORDER_KEYS_TEST], dtype=np.float64
)

print(f"Ticker: {ticker_symbol}")
print(f"Test samples: {test_sample_idx.size:,}  (orders: {n_test:,})")
print(f"feature_dim={FEATURE_DIM_TOTAL}, seq_len={SEQUENCE_LENGTH}, output_steps={output_steps}")
print(f"CIF baseline array: {2 * output_steps * test_sample_idx.size * 4 / 1e6:.1f} MB")
print(f"Feature strip per permutation: {test_sample_idx.size * SEQUENCE_LENGTH * 4 / 1e9:.2f} GB")


## Model Helpers

In [5]:
NUM_COMPETING_EVENTS = 2
EVENT_NAMES = ["FAVORABLE_FILL", "TOXIC_FILL"]
EVENT_CODES = [1, 2]


def _build_model(model_name: str) -> torch.nn.Module:
    feat_dim = FEATURE_DIM_TOTAL
    seq_len  = SEQUENCE_LENGTH
    T        = output_steps
    K        = NUM_COMPETING_EVENTS
    if model_name == "gru":
        return DeepHitRNNCompeting(
            num_features=feat_dim, num_events=K, num_time_steps=T,
            hidden_size=160, num_layers=2, rnn_dropout=0.2,
            fc_hidden=int(160 * 1.75), fc_dropout=0.2,
        )
    elif model_name == "gru_transformer":
        return DeepHitRNNTransformerCompeting(
            num_features=feat_dim, num_events=K, num_time_steps=T,
            hidden_size=96, num_layers=2, rnn_dropout=0.2,
            transformer_layers=1, transformer_heads=2,
            transformer_ff_dim=int(96 * 2.0), transformer_dropout=0.1,
            max_seq_len=seq_len, fc_hidden=int(96 * 1.75), fc_dropout=0.2,
        )
    elif model_name == "transformer":
        return DeepHitTransformerCompeting(
            num_features=feat_dim, num_events=K, num_time_steps=T,
            hidden_size=96, num_layers=2, num_heads=4,
            transformer_ff_dim=int(96 * 2.0), transformer_dropout=0.1,
            max_seq_len=seq_len, fc_hidden=int(96 * 1.75), fc_dropout=0.2,
        )
    elif model_name == "mamba":
        return DeepHitMambaCompeting(
            num_features=feat_dim, num_events=K, num_time_steps=T,
            hidden_size=144, num_mamba_layers=1, d_state=8, d_conv=4, expand=2,
            mamba_dropout=0.15, fc_hidden=int(144 * 1.75), fc_dropout=0.2,
        )
    raise ValueError(f"Unknown model_name: {model_name}")


def load_model(model_name: str) -> torch.nn.Module | None:
    """Load trained weights from artifact directory. Returns None if artifact missing."""
    path = Path(ARTIFACT_DIR) / ticker_symbol / f"{model_name}_{ticker_symbol}.pt"
    if not path.exists():
        return None
    state_dict = torch.load(path, map_location="cpu", weights_only=True)
    model = _build_model(model_name).to(device)
    model.load_state_dict(state_dict)
    model.eval()
    return model


## Streaming Inference and Permutation

In [6]:
def _stream_inference(
    model: torch.nn.Module,
    sample_idx: np.ndarray,
    feature_strip_j: np.ndarray | None = None,
    feature_j: int | None = None,
) -> np.ndarray:
    """Stream through all test samples in batches and collect CIF.

    If feature_strip_j is provided, column feature_j of each batch is replaced
    with the corresponding rows from feature_strip_j (already permuted).

    Returns CIF of shape (K, T, N).
    """
    N = len(sample_idx)
    K = NUM_COMPETING_EVENTS
    T = output_steps
    cif = np.empty((K, T, N), dtype=np.float32)

    with torch.no_grad():
        for start in range(0, N, BATCH_SIZE):
            batch_idx = sample_idx[start : start + BATCH_SIZE]
            x_b, *_ = materialize_dynamic_samples_from_manifest(
                ORDER_STORE, SAMPLE_MANIFEST, batch_idx
            )
            x_b = apply_dynamic_normalizer(x_b, feat_mean, feat_std)

            if feature_strip_j is not None:
                x_b[:, :, feature_j] = feature_strip_j[start : start + len(batch_idx)]

            x_t = torch.from_numpy(x_b).to(device)
            logits = model(x_t)  # (B, K*T)
            B = logits.size(0)
            pmf = torch.softmax(logits.reshape(B, -1), dim=1).reshape(B, K, T)
            cif_b = torch.cumsum(pmf, dim=2).cpu().numpy()  # (B, K, T)

            end = start + B
            cif[:, :, start:end] = np.transpose(cif_b, (1, 2, 0))

    return cif


def _extract_feature_strip(sample_idx: np.ndarray, j: int) -> np.ndarray:
    """Stream through test samples and extract column j after normalisation.
    Returns array of shape (N, seq_len).
    """
    N = len(sample_idx)
    strip = np.empty((N, SEQUENCE_LENGTH), dtype=np.float32)
    for start in range(0, N, BATCH_SIZE):
        batch_idx = sample_idx[start : start + BATCH_SIZE]
        x_b, *_ = materialize_dynamic_samples_from_manifest(
            ORDER_STORE, SAMPLE_MANIFEST, batch_idx
        )
        x_b = apply_dynamic_normalizer(x_b, feat_mean, feat_std)
        strip[start : start + len(batch_idx)] = x_b[:, :, j]
    return strip


## Metric Helpers

In [7]:
def _dynamic_weighted_ctd(
    cif_event: np.ndarray,   # (T, N)
    durations: np.ndarray,
    events: np.ndarray,
    event_code: int,
    update_idx: np.ndarray,
    weights: np.ndarray,
    eps: float = 1e-12,
) -> float:
    tau_idx = np.clip(
        np.searchsorted(time_grid, durations, side="left"), 0, len(time_grid) - 1
    )
    num = den = 0.0
    for step in np.unique(update_idx):
        step_idx = np.where(update_idx == step)[0]
        if step_idx.size <= 1:
            continue
        anchors = step_idx[events[step_idx] == event_code]
        if anchors.size == 0:
            continue
        dur_step = durations[step_idx]
        pos_map  = {int(idx): pos for pos, idx in enumerate(step_idx.tolist())}
        for i in anchors:
            i_pos = pos_map[int(i)]
            later_mask = dur_step > dur_step[i_pos]
            if not np.any(later_mask):
                continue
            later_idx  = step_idx[later_mask]
            s_i = float(cif_event[int(tau_idx[i]), i])
            s_j = cif_event[int(tau_idx[i]), later_idx]
            w_pair = weights[i] * weights[later_idx]
            concordant = (s_i > s_j).astype(np.float64)
            ties       = (s_i == s_j).astype(np.float64)
            num += float(np.sum(w_pair * (concordant + 0.5 * ties)))
            den += float(np.sum(w_pair))
    return num / den if den > eps else float("nan")


def _safe_pr_auc(
    y_true: np.ndarray, y_score: np.ndarray, sample_weight=None
) -> float:
    if int(y_true.sum()) == 0:
        return float("nan")
    precision, recall, _ = precision_recall_curve(
        y_true, y_score, sample_weight=sample_weight
    )
    return float(auc(recall[::-1], precision[::-1]))


def compute_metrics(cif: np.ndarray) -> dict:
    """Compute class-weighted C-td and weighted PR-AUC for TOXIC_FILL.
    cif: (K, T, N)
    Returns the two primary importance metrics.
    """
    ctd_fav = _dynamic_weighted_ctd(
        cif[0], Y_test, D_test, 1, UPDATE_IDX_TEST, ORDER_TEST_WEIGHTS
    )
    ctd_tox = _dynamic_weighted_ctd(
        cif[1], Y_test, D_test, 2, UPDATE_IDX_TEST, ORDER_TEST_WEIGHTS
    )
    n_fav = int((D_test == 1).sum())
    n_tox = int((D_test == 2).sum())
    n_ev  = n_fav + n_tox
    ctd_weighted = (
        (n_fav * ctd_fav + n_tox * ctd_tox) / n_ev if n_ev > 0 else float("nan")
    )
    y_toxic = (D_test == 2).astype(int)
    pr_auc_toxic_weighted = _safe_pr_auc(y_toxic, cif[1, -1, :], ORDER_TEST_WEIGHTS)
    return {
        "ctd_weighted": ctd_weighted,
        "pr_auc_toxic_weighted": pr_auc_toxic_weighted,
    }


In [ ]:
# W&B Setup
dotenv_path = find_dotenv(filename=".env", usecwd=True)
if dotenv_path:
    load_dotenv(dotenv_path=dotenv_path, override=False)
else:
    load_dotenv(override=False)

WANDB_ENABLE = True
WANDB_PROJECT = "toxicity-ablation"
WANDB_MODE = "online"
WANDB_ENTITY = os.environ.get("WANDB_ENTITY", "").strip()
WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "").strip()
WANDB_RESUME_RUN_ID = os.environ.get("WANDB_RESUME_RUN_ID", "").strip()

if WANDB_RESUME_RUN_ID:
    WANDB_RUN_NAME = None
    WANDB_RUN_ID = WANDB_RESUME_RUN_ID
else:
    WANDB_RUN_NAME = f"toxicity_ablation_{TICKER}_{MODEL_NAME}"
    WANDB_RUN_ID = None

wandb_run = None
if WANDB_ENABLE:
    if not WANDB_ENTITY:
        raise ValueError("WANDB_ENTITY is missing. Set it in .env (example: WANDB_ENTITY=your_wandb_team_or_username).")
    if not WANDB_API_KEY:
        raise ValueError("WANDB_API_KEY is missing. Set it in .env with your W&B API key.")

    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    wandb.login(key=WANDB_API_KEY, relogin=True)

    wandb_config = {
        "ticker": TICKER,
        "model_name": MODEL_NAME,
        "ablation_seed": ABLATION_SEED,
        "preprocessing_contract_version": PREPROCESSING_CONTRACT_VERSION,
        "max_samples_per_order": MAX_SAMPLES_PER_ORDER,
        "split_cap_random_seed": SPLIT_CAP_RANDOM_SEED,
        "num_time_steps": NUM_TIME_STEPS,
        "batch_size": BATCH_SIZE,
        "num_ablation_features": len(ABLATION_FEATURE_NAMES),
    }

    init_kwargs = {
        "project": WANDB_PROJECT,
        "entity": WANDB_ENTITY,
        "config": wandb_config,
        "mode": WANDB_MODE,
        "reinit": False,
    }

    if WANDB_RESUME_RUN_ID:
        init_kwargs["id"] = WANDB_RESUME_RUN_ID
        init_kwargs["resume"] = "allow"
        print(f"Resuming W&B run: {WANDB_RESUME_RUN_ID}")
    else:
        init_kwargs["name"] = WANDB_RUN_NAME
        print(f"Starting new W&B run: {WANDB_RUN_NAME}")

    wandb_run = wandb.init(**init_kwargs)
    print(f"W&B initialized: entity={WANDB_ENTITY}, project={WANDB_PROJECT}, run_id={wandb_run.id}")
else:
    print("W&B logging is disabled for this run.")


## Permutation Importance Runner

In [8]:
def run_permutation_importance(model: torch.nn.Module) -> dict:
    """Run permutation importance for the 13 toxicity features.

    For each feature in ABLATION_FEATURE_NAMES:
      1. Extract its normalised strip across all N test samples  (N, seq_len)
      2. Shuffle the strip (permute samples)
      3. Re-run streaming inference with the shuffled column injected
      4. importance[feat] = baseline_metric - permuted_metric

    Returns:
      baseline   : dict metric_name -> float
      importance : dict feature_name -> dict metric_name -> float (positive = important)
    """
    print("  Computing baseline CIF...")
    cif_base = _stream_inference(model, test_sample_idx)
    baseline = compute_metrics(cif_base)
    del cif_base
    gc.collect()

    print(
        f"  Baseline: ctd_weighted={baseline['ctd_weighted']:.4f}  "
        f"pr_auc_toxic_weighted={baseline['pr_auc_toxic_weighted']:.4f}"
    )

    importance: dict[str, dict] = {}
    n_features = len(ABLATION_FEATURE_NAMES)

    for step, feat_name in enumerate(ABLATION_FEATURE_NAMES):
        j = ALL_FEATURE_NAMES.index(feat_name)  # column index in the full feature tensor
        print(f"  [{step+1:2d}/{n_features}] Permuting '{feat_name}' (col {j})...", end=" ", flush=True)

        strip_j = _extract_feature_strip(test_sample_idx, j)
        perm = rng.permutation(len(test_sample_idx))
        strip_j = strip_j[perm]  # shuffle across samples

        cif_perm = _stream_inference(model, test_sample_idx, strip_j, j)
        perm_metrics = compute_metrics(cif_perm)

        importance[feat_name] = {
            metric: baseline[metric] - perm_metrics[metric]
            for metric in baseline
        }
        print(
            f"Δctd_weighted={importance[feat_name]['ctd_weighted']:+.4f}  "
            f"Δpr_auc_toxic_weighted={importance[feat_name]['pr_auc_toxic_weighted']:+.4f}"
        )

        del strip_j, cif_perm
        gc.collect()

    return {"baseline": baseline, "importance": importance}


## Run Ablation for All Model Variants

In [9]:
all_results: list[dict] = []

artifact_path = Path(ARTIFACT_DIR) / ticker_symbol / f"{MODEL_NAME}_{ticker_symbol}.pt"
if not artifact_path.exists():
    print(f"[{MODEL_NAME}] No artifact at {artifact_path}, exiting.")
    print("\nNo artifacts found to evaluate.")
    exit(1)

print(f"\n{'='*60}")
print(f"Model: {MODEL_NAME}")
print(f"{'='*60}")
model = load_model(MODEL_NAME)

result = run_permutation_importance(model)

for feat_name, imp in result["importance"].items():
    feat_idx = ALL_FEATURE_NAMES.index(feat_name)
    all_results.append({
        "ticker":        ticker_symbol,
        "model":         MODEL_NAME,
        "feature":       feat_name,
        "feature_idx":   feat_idx,
        "feature_group": FEATURE_GROUPS[feat_idx],
        "baseline_ctd_weighted":          result["baseline"]["ctd_weighted"],
        "baseline_pr_auc_toxic_weighted":  result["baseline"]["pr_auc_toxic_weighted"],
        "imp_ctd_weighted":               imp["ctd_weighted"],
        "imp_pr_auc_toxic_weighted":      imp["pr_auc_toxic_weighted"],
    })

del model
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()

if not all_results:
    raise RuntimeError("No results collected. Check TICKER and MODEL_NAME.")
print("\nAblation complete.")


## Results: Feature Importance Table

In [ ]:
df = pd.DataFrame(all_results)

# Aggregate across model variants (mean importance)
agg = (
    df.groupby(["feature", "feature_idx", "feature_group"], as_index=False)
    .agg(
        imp_ctd_weighted=("imp_ctd_weighted", "mean"),
        imp_pr_auc_toxic_weighted=("imp_pr_auc_toxic_weighted", "mean"),
        n_models=("model", "nunique"),
    )
    .sort_values("imp_ctd_weighted", ascending=False)
    .reset_index(drop=True)
)
agg["rank"] = range(1, len(agg) + 1)

print(f"Feature importance for {ticker_symbol} (aggregated over {agg['n_models'].iloc[0]} model(s))")
print("Positive = metric drops when feature is permuted (feature is important)")
print()
print(
    agg[["rank", "feature", "feature_group", "imp_ctd_weighted", "imp_pr_auc_toxic_weighted", "n_models"]]
    .to_string(index=False)
)


## Per-Model Breakdown

In [ ]:
for model_name, grp in df.groupby("model"):
    grp_sorted = grp.sort_values("imp_ctd_weighted", ascending=False)
    baseline_ctd = grp_sorted["baseline_ctd_weighted"].iloc[0]
    baseline_pr  = grp_sorted["baseline_pr_auc_toxic_weighted"].iloc[0]
    print(f"\n--- {model_name}  (baseline ctd_weighted={baseline_ctd:.4f}, pr_auc_toxic_weighted={baseline_pr:.4f}) ---")
    print(
        grp_sorted.head(10)[["feature", "feature_group", "imp_ctd_weighted", "imp_pr_auc_toxic_weighted"]]
        .to_string(index=False)
    )


## Visualisation

In [ ]:
group_colors = {"LOB": "#4e79a7", "Toxicity": "#f28e2b", "Meta": "#76b7b2"}

fig, axes = plt.subplots(1, 2, figsize=(16, max(6, len(ALL_FEATURE_NAMES) * 0.38)))

for ax, metric_col, xlabel, title in [
    (axes[0], "imp_ctd_weighted",         "Mean Class-Weighted C-td Drop",          "Class-Weighted C-td"),
    (axes[1], "imp_pr_auc_toxic_weighted", "Mean Weighted PR-AUC Drop (TOXIC_FILL)", "Weighted PR-AUC (Toxic)"),
]:
    agg_s = agg.sort_values(metric_col, ascending=True)
    colors = [group_colors.get(g, "#999999") for g in agg_s["feature_group"]]
    ax.barh(agg_s["feature"], agg_s[metric_col], color=colors)
    ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_xlabel(xlabel)
    ax.set_title(f"Permutation Importance\n{title} drop")
    ax.grid(axis="x", alpha=0.3)

from matplotlib.patches import Patch
legend_patches = [Patch(facecolor=c, label=g) for g, c in group_colors.items()]
fig.legend(handles=legend_patches, loc="upper right", title="Feature Group")
fig.suptitle(
    f"Permutation Feature Importance — {ticker_symbol}  "
    f"({df['model'].nunique()} model(s): {', '.join(df['model'].unique())})",
    fontsize=13,
)
plt.tight_layout()
plt.show()


## Heatmap: Importance per Model

In [ ]:
pivot = (
    df.pivot_table(
        index="feature", columns="model",
        values="imp_ctd_weighted", aggfunc="mean"
    )
)
# Sort rows by mean importance
pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(
    figsize=(max(6, len(pivot.columns) * 1.5), max(8, len(pivot) * 0.4))
)
vabs = float(np.nanpercentile(np.abs(pivot.values), 95))
im = ax.imshow(pivot.values, aspect="auto", cmap="RdYlGn", vmin=-vabs, vmax=vabs)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=30, ha="right")
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
plt.colorbar(im, ax=ax, label="Class-weighted C-td drop")
ax.set_title(
    f"{ticker_symbol} — Permutation Importance per Model\n"
    "(green = important, red = harmful when permuted)"
)
plt.tight_layout()
plt.show()


## Save Results

In [ ]:
results_dir = repo_root / "results" / "feature_ablation"
results_dir.mkdir(parents=True, exist_ok=True)

detail_path = results_dir / f"feature_importance_{ticker_symbol}_{MODEL_NAME}.csv"
agg_path    = results_dir / f"feature_importance_{ticker_symbol}_{MODEL_NAME}_aggregated.csv"

df.to_csv(detail_path, index=False)
agg.to_csv(agg_path, index=False)

print(f"Saved: {detail_path}")
print(f"Saved: {agg_path}")
print(f"\nTop 10 most important features ({ticker_symbol}, class-weighted C-td drop):")
print(
    agg.head(10)[["rank", "feature", "feature_group", "imp_ctd_weighted", "imp_pr_auc_toxic_weighted"]]
    .to_string(index=False)
)

# Log to W&B if enabled
if wandb_run is not None:
    wandb_run.log({
        "num_features_tested": len(ABLATION_FEATURE_NAMES),
        "baseline_ctd_weighted": float(df["baseline_ctd_weighted"].iloc[0]) if len(df) > 0 else None,
        "baseline_pr_auc_toxic_weighted": float(df["baseline_pr_auc_toxic_weighted"].iloc[0]) if len(df) > 0 else None,
    })
    
    for _, row in agg.head(10).iterrows():
        wandb_run.log({
            f"feature_importance/{row['feature']}_ctd_weighted": float(row["imp_ctd_weighted"]),
            f"feature_importance/{row['feature']}_pr_auc_toxic_weighted": float(row["imp_pr_auc_toxic_weighted"]),
        })
    
    wandb_run.summary["output_detail_csv"] = detail_path
    wandb_run.summary["output_agg_csv"] = agg_path
    print("Logged results to W&B.")
